In [15]:
import pandas as pd
import numpy as np

# =========================
# Load Data
# =========================

df = pd.read_csv("Downloads/Nifty 50 Historical Data.csv")

# Convert Date column
df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True,
    format="mixed"
)

# Sort oldest → newest
df = df.sort_values("Date").reset_index(drop=True)

# Clean Price column
df["Price"] = (
    df["Price"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype(float)
)

close = df["Price"]

# =========================
# Bollinger Bands (20,2)
# =========================

window = 20

ma20 = close.rolling(window).mean()
std20 = close.rolling(window).std()

df["UpperBB"] = ma20 + 2 * std20
df["LowerBB"] = ma20 - 2 * std20

# =========================
# RSI (14)
# =========================

delta = close.diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df["RSI"] = 100 - (100 / (1 + rs))

# =========================
# Trading Logic
# =========================

holding = 0           # 0 = no stock, 1 = holding one lot
entry_price = None

signals = []

STOP_LOSS = 0.03
TAKE_PROFIT = 0.05

for i in range(len(df)):

    signal = 0

    if (
        pd.isna(df.loc[i, "RSI"])
        or pd.isna(df.loc[i, "UpperBB"])
    ):
        signals.append(0)
        continue

    price = df.loc[i, "Price"]
    rsi = df.loc[i, "RSI"]
    upper_bb = df.loc[i, "UpperBB"]
    lower_bb = df.loc[i, "LowerBB"]

    # ==================================
    # BUY
    # ==================================
    if holding == 0:

        buy_condition = (
            price < lower_bb and
            rsi < 30
        )

        if buy_condition:

            signal = 1
            holding = 1
            entry_price = price

    # ==================================
    # SELL
    # ==================================
    elif holding == 1:

        take_profit_hit = (
            price >= entry_price * (1 + TAKE_PROFIT)
        )

        stop_loss_hit = (
            price <= entry_price * (1 - STOP_LOSS)
        )

        indicator_exit = (
            price > upper_bb and
            rsi > 70
        )

        if (
            take_profit_hit
            or stop_loss_hit
            or indicator_exit
        ):

            signal = -1
            holding = 0
            entry_price = None

    signals.append(signal)

# =========================
# Save Output
# =========================

result = pd.DataFrame({
    "Date": df["Date"].dt.strftime("%Y-%m-%d"),
    "Signal": signals
})

result.to_csv(
    "bollinger_rsi_trades.csv",
    index=False
)

print(result.head())
print("\nSaved as bollinger_rsi_trades.csv")

         Date  Signal
0  2016-01-01       0
1  2016-01-04       0
2  2016-01-05       0
3  2016-01-06       0
4  2016-01-07       0

Saved as bollinger_rsi_trades.csv
